# System-matrix recomputation mode

This notebook tests the `Auto`, `Enabled`, and `Disabled` system-matrix
recomputation modes with two small DP simulations:

1. A non-switch variable component: `AvVoltSourceInverterStateSpace`.
   `Auto` must select recomputation and produce the same result as `Enabled`.
2. A switch-only circuit. `Auto` must keep the precomputed-matrix path and
   produce the same result as `Disabled`.

It also checks that explicitly selecting `Disabled` for the variable component
does not stop the simulation and writes the expected warning.


In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

import dpsimpy


Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver
Mode = dpsimpy.SystemMatrixRecomputationMode

sp = dpsimpy.sp
dp = dpsimpy.dp
sp_ph1 = sp.ph1
dp_ph1 = dp.ph1

log_root = Path("logs/SystemMatrixRecomputationMode")

system_frequency = 50.0
system_omega = 2.0 * np.pi * system_frequency
time_step = 100e-6
final_time = 1e-3


def prepare_run(name):
    run_dir = log_root / name
    shutil.rmtree(run_dir, ignore_errors=True)
    Logger.set_log_dir(str(run_dir))
    return run_dir


def read_result(run_dir, name):
    path = run_dir / f"{name}.csv"
    assert path.exists(), f"Missing simulation result: {path}"
    return pd.read_csv(path, skipinitialspace=True)


def read_log_text(run_dir):
    parts = []
    for path in sorted(run_dir.rglob("*")):
        if path.is_file() and path.suffix != ".csv":
            parts.append(path.read_text(encoding="utf-8", errors="ignore"))
    return "\n".join(parts)


def assert_log_contains(run_dir, text):
    log_text = read_log_text(run_dir)
    assert text in log_text, f"Expected log message not found: {text!r}"


def assert_results_equal(first, second, label):
    assert list(first.columns) == list(second.columns)
    np.testing.assert_allclose(
        first.to_numpy(),
        second.to_numpy(),
        rtol=1e-10,
        atol=1e-12,
        err_msg=label,
    )

## 1. Non-switch variable component

A short power flow provides the initial node voltages. The dynamic simulation
then contains a grid, a line, and the variable state-space inverter.


In [ ]:
grid_voltage = 400.0

line_resistance = 0.3
line_inductance = 0.1e-3
line_capacitance = 1e-6
line_conductance = 1e-6

lf = 2e-3
cf = 10e-6
rf = 0.2
rc = 0.2

p_ref = 10_000.0
q_ref = 5_000.0

kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = system_omega
kp_power_ctrl = 0.05
ki_power_ctrl = 0.2
kp_curr_ctrl = 0.25
ki_curr_ctrl = 1.0


def build_power_flow():
    name = "MatrixMode_PF"
    prepare_run(name)

    n_grid = sp.SimNode("nGrid", PhaseType.Single)
    n_pcc = sp.SimNode("nPcc", PhaseType.Single)

    slack = sp_ph1.NetworkInjection("Slack")
    slack.set_parameters(grid_voltage)
    slack.set_base_voltage(grid_voltage)
    slack.modify_power_flow_bus_type(PowerflowBusType.VD)

    line = sp_ph1.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )
    line.set_base_voltage(grid_voltage)

    inverter_injection = sp_ph1.Load("INV_SSN_PLL")
    inverter_injection.set_parameters(-p_ref, -q_ref, grid_voltage)
    inverter_injection.modify_power_flow_bus_type(PowerflowBusType.PQ)

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])
    inverter_injection.connect([n_pcc])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter_injection],
    )

    sim = Simulation(name, dpsimpy.LogLevel.info)
    sim.set_system(system)
    sim.set_domain(Domain.SP)
    sim.set_solver(Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim.do_init_from_nodes_and_terminals(False)
    sim.set_time_step(1.0)
    sim.set_final_time(2.0)
    sim.run()

    return system


system_pf = build_power_flow()


def run_variable_case(name, mode=None):
    run_dir = prepare_run(name)

    n_grid = dp.SimNode("nGrid", PhaseType.Single)
    n_pcc = dp.SimNode("nPcc", PhaseType.Single)

    slack = dp_ph1.NetworkInjection("Slack")

    line = dp_ph1.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )

    inverter = dp_ph1.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        system_omega,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        kp_power_ctrl,
        ki_power_ctrl,
        kp_curr_ctrl,
        ki_curr_ctrl,
    )

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])
    inverter.connect([dp.SimNode.gnd, n_pcc])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter],
    )
    system.init_with_powerflow(system_pf, Domain.DP)

    logger = Logger(name)
    logger.log_attribute("v_pcc", n_pcc.attr("v"))
    logger.log_attribute("p_inst", inverter.attr("p_inst"))

    sim = Simulation(name, dpsimpy.LogLevel.info)
    sim.set_system(system)
    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)
    sim.do_init_from_nodes_and_terminals(True)
    sim.add_logger(logger)

    if mode is not None:
        sim.set_system_matrix_recomputation_mode(mode)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

    return run_dir, read_result(run_dir, name)

In [ ]:
variable_auto_dir, variable_auto = run_variable_case("MatrixMode_Variable_Auto")
variable_enabled_dir, variable_enabled = run_variable_case(
    "MatrixMode_Variable_Enabled",
    Mode.Enabled,
)
variable_disabled_dir, variable_disabled = run_variable_case(
    "MatrixMode_Variable_Disabled",
    Mode.Disabled,
)

assert_log_contains(
    variable_auto_dir,
    "System-matrix recomputation enabled automatically for",
)
assert_log_contains(variable_auto_dir, "INV_SSN_PLL")
assert_log_contains(
    variable_enabled_dir,
    "System-matrix recomputation enabled.",
)
assert_log_contains(
    variable_disabled_dir,
    "System-matrix recomputation disabled, but",
)
assert_log_contains(variable_disabled_dir, "INV_SSN_PLL")

assert len(variable_disabled) > 1
assert_results_equal(
    variable_auto,
    variable_enabled,
    "Auto and Enabled differ for the variable component.",
)

print("Variable-component checks passed.")

## 2. Switch-only circuit

The circuit contains a voltage source, a switch, and a resistor. Closing the
switch creates a small transient. Because the only variable component is a
normal switch, `Auto` should retain the precomputed switch matrices.


In [ ]:
def run_switch_case(name, mode=None):
    run_dir = prepare_run(name)

    gnd = dp.SimNode.gnd
    n_source = dp.SimNode("nSource", PhaseType.Single)
    n_load = dp.SimNode("nLoad", PhaseType.Single)

    source = dp_ph1.VoltageSource("Source")
    source.set_parameters(
        V_ref=complex(325.0, 0.0),
        f_src=system_frequency,
    )

    switch = dp_ph1.Switch("Switch")
    switch.set_parameters(1e12, 1e-6, False)

    load = dp_ph1.Resistor("Load")
    load.set_parameters(10.0)

    source.connect([gnd, n_source])
    switch.connect([n_source, n_load])
    load.connect([n_load, gnd])

    system = SystemTopology(
        system_frequency,
        [gnd, n_source, n_load],
        [source, switch, load],
    )

    logger = Logger(name)
    logger.log_attribute("v_load", n_load.attr("v"))
    logger.log_attribute("i_load", load.attr("i_intf"))

    sim = Simulation(name, dpsimpy.LogLevel.info)
    sim.set_system(system)
    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)
    sim.do_init_from_nodes_and_terminals(False)
    sim.add_logger(logger)

    if mode is not None:
        sim.set_system_matrix_recomputation_mode(mode)

    sim.add_event(dpsimpy.event.SwitchEvent(5.0 * time_step, switch, True))
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

    return run_dir, read_result(run_dir, name)


switch_auto_dir, switch_auto = run_switch_case("MatrixMode_Switch_Auto")
switch_disabled_dir, switch_disabled = run_switch_case(
    "MatrixMode_Switch_Disabled",
    Mode.Disabled,
)

assert_log_contains(
    switch_auto_dir,
    "System-matrix recomputation disabled automatically.",
)
assert_log_contains(
    switch_disabled_dir,
    "System-matrix recomputation disabled.",
)
assert_results_equal(
    switch_auto,
    switch_disabled,
    "Auto and Disabled differ for the switch-only circuit.",
)

print("Switch-only checks passed.")